# Diabetes Health Indicators: Deep Learning Classification

## Deep Learning Application in Data Science

**Project Type:** Binary Classification  
**Framework:** PyTorch  
**Dataset:** CDC Diabetes Health Indicators  
**Source:** UCI Machine Learning Repository  
**Hardware:** CPU

---

## Project Objective

The objective of this project is to develop a deep learning model that predicts whether an individual belongs to the diabetes class using health, lifestyle, demographic, and socioeconomic indicators.

### Workflow

1. Dataset acquisition
2. Data understanding
3. Exploratory Data Analysis
4. Duplicate removal
5. Data preprocessing
6. Train-validation-test splitting
7. Feature standardization
8. Neural network design
9. Class imbalance handling
10. Model training
11. Model evaluation
12. Threshold analysis
13. Critical analysis
14. Conclusion

> **Academic Disclaimer:** This project is for educational and data-science purposes and is not intended for medical diagnosis or clinical decision-making.

# 1. Problem Statement

Diabetes classification is formulated as a supervised binary classification problem.

The neural network receives health-related indicators as input and predicts whether the target class corresponds to diabetes.

## Target Variable

| Value | Meaning |
|---:|---|
| 0 | No Diabetes |
| 1 | Diabetes |

## Objective

A feed-forward Multilayer Perceptron (MLP) is used to learn nonlinear relationships between the input features and the binary target.

# 2. Dataset Description

The project uses the Diabetes Health Indicators Dataset, based on CDC BRFSS 2015 data and distributed through the UCI Machine Learning Repository.

## Selected Dataset

`diabetes_binary_health_indicators_BRFSS2015.csv`

## Dataset Characteristics

- Original records: **253,680**
- Input features: **21**
- Target variables: **1**
- Total columns: **22**
- Missing values: **0**
- Duplicate rows: **24,206**

The selected dataset is the binary classification version.

# 3. Import Libraries

In [ ]:
import os
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

import torch
import torch.nn as nn

print('PyTorch version:', torch.__version__)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', device)

# 4. Load Dataset

The dataset is loaded from the project's data directory.

In [ ]:
DATA_PATH = (
    '../data/'
    'diabetes_binary_health_indicators_BRFSS2015.csv'
)

df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)

display(df.head())

# 5. Data Understanding

The initial inspection examines dataset dimensions, column names, data types, missing values, duplicate records, and target distribution.

In [ ]:
print('=' * 60)
print('DATASET INFORMATION')
print('=' * 60)

print('\nShape:')
print(df.shape)

print('\nColumns:')
print(df.columns.tolist())

print('\nData Types:')
print(df.dtypes)

print('\nMissing Values:')
print(df.isnull().sum().sum())

print('\nDuplicate Rows:')
print(df.duplicated().sum())

print('\nTarget Distribution:')
print(df['Diabetes_binary'].value_counts())

# 6. Exploratory Data Analysis

## Target Distribution

The target distribution is examined because the dataset contains an imbalance between the No Diabetes and Diabetes classes.

In [ ]:
target_counts = (
    df['Diabetes_binary']
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 5))

plt.bar(
    ['No Diabetes', 'Diabetes'],
    target_counts.values
)

plt.title('Diabetes Target Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Records')
plt.tight_layout()
plt.show()

## Feature Correlation with Diabetes

Pearson correlation is used to examine linear association between individual numerical features and the target.

**Correlation does not imply causation.**

In [ ]:
correlations = (
    df.corr(numeric_only=True)['Diabetes_binary']
    .drop('Diabetes_binary')
    .sort_values(key=abs, ascending=False)
)

print('Top 10 feature correlations:')
display(
    correlations.head(10).to_frame('Correlation')
)

In [ ]:
plt.figure(figsize=(13, 9))

sns.heatmap(
    df.corr(numeric_only=True),
    cmap='coolwarm',
    center=0
)

plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# 7. Duplicate Removal

The original dataset contains **24,206 exact duplicate rows**.

Exact duplicate rows are removed before the train, validation, and test split.

In [ ]:
original_rows = len(df)
duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print('Original rows:', original_rows)
print('Duplicates removed:', duplicate_count)
print('Rows after duplicate removal:', len(df))

# 8. Feature and Target Separation

In [ ]:
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

print('Features:', X.shape)
print('Target:', y.shape)

print('\nTarget distribution after duplicate removal:')
print(y.value_counts())

# 9. Train / Validation / Test Split

The cleaned dataset is divided into 70% training, 15% validation, and 15% testing.

Stratification is used to preserve class proportions.

Random seed: **42**

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print('Training:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

# 10. Feature Standardization

StandardScaler is fitted only on the training data and then applied to validation and test data.

This prevents information leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Training scaled:', X_train_scaled.shape)
print('Validation scaled:', X_val_scaled.shape)
print('Test scaled:', X_test_scaled.shape)

# 11. Convert Data to PyTorch Tensors

In [ ]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).view(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

print('Tensor conversion completed.')

# 12. Neural Network Architecture

The project uses a **Multilayer Perceptron (MLP)**.

## Network Structure

```text
21 Input Features
       |
       v
Linear 21 -> 64
       |
BatchNorm1d
       |
ReLU
       |
Dropout 0.30
       |
Linear 64 -> 32
       |
BatchNorm1d
       |
ReLU
       |
Dropout 0.20
       |
Linear 32 -> 16
       |
ReLU
       |
Linear 16 -> 1
       |
       v
Diabetes Probability
```

## Design Justification

- 21 input features match the dataset.
- 64, 32, and 16 neurons progressively compress the representation.
- ReLU provides nonlinear transformations.
- Batch normalization improves training stability.
- Dropout provides regularization.
- A single output neuron is appropriate for binary classification.
- BCEWithLogitsLoss provides numerically stable binary classification loss.

In [ ]:
class DiabetesMLP(nn.Module):

    def __init__(self, input_features=21):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


model = DiabetesMLP()

print(model)

# 13. Training Configuration

| Parameter | Value |
|---|---:|
| Optimizer | AdamW |
| Learning Rate | 0.001 |
| Weight Decay | 0.0001 |
| Batch Size | 1024 |
| Maximum Epochs | 30 |
| Early Stopping Patience | 7 |
| Dropout | 0.30 / 0.20 |
| Random Seed | 42 |
| Device | CPU |
| Positive Class Weight | 5.5382 |

The positive class weight was used to address the class imbalance.

# 14. Training Results

The model was trained with early stopping.

The best validation checkpoint was obtained at **epoch 13**.

## Best Validation Results

| Metric | Result |
|---|---:|
| Validation Loss | **0.8748** |
| Accuracy | **70.72%** |
| Precision | **31.76%** |
| Recall | **79.60%** |
| F1 Score | **45.40%** |
| ROC-AUC | **81.95%** |

In [ ]:
history = pd.read_csv(
    '../models/training_history.csv'
)

display(history)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history['epoch'],
    history['train_loss'],
    marker='o',
    label='Training Loss'
)

plt.plot(
    history['epoch'],
    history['val_loss'],
    marker='o',
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history['epoch'],
    history['accuracy'],
    marker='o',
    label='Accuracy'
)

plt.plot(
    history['epoch'],
    history['precision'],
    marker='o',
    label='Precision'
)

plt.plot(
    history['epoch'],
    history['recall'],
    marker='o',
    label='Recall'
)

plt.plot(
    history['epoch'],
    history['f1'],
    marker='o',
    label='F1 Score'
)

plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('Validation Classification Metrics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 15. Test-Set Evaluation

The best saved model was evaluated on the unseen test dataset.

## Final Test Metrics

| Metric | Result |
|---|---:|
| Accuracy | **70.48%** |
| Precision | **31.54%** |
| Recall | **79.49%** |
| F1 Score | **45.17%** |
| ROC-AUC | **81.67%** |

The primary evaluation threshold is **0.50**.

In [ ]:
test_results = pd.read_csv(
    '../models/test_results.csv'
)

display(test_results)

## Classification Report

In [ ]:
predictions = pd.read_csv(
    '../models/test_predictions.csv'
)

y_true_test = predictions['actual']
y_pred_test = predictions['prediction']

print(
    classification_report(
        y_true_test,
        y_pred_test,
        target_names=['No Diabetes', 'Diabetes'],
        digits=4
    )
)

# 16. Confusion Matrix

At the 0.50 threshold:

| | Predicted No Diabetes | Predicted Diabetes |
|---|---:|---:|
| Actual No Diabetes | **20,075** | **9,082** |
| Actual Diabetes | **1,080** | **4,185** |

### Components

- True Negatives = **20,075**
- False Positives = **9,082**
- False Negatives = **1,080**
- True Positives = **4,185**

In [ ]:
cm = confusion_matrix(
    y_true_test,
    y_pred_test
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Diabetes', 'Diabetes'],
    yticklabels=['No Diabetes', 'Diabetes']
)

plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# 17. ROC Curve

The model achieved a test ROC-AUC of **0.8167**.

ROC-AUC measures the model's ability to distinguish between the two classes across different thresholds.

In [ ]:
y_probability_test = predictions['probability']

fpr, tpr, thresholds = roc_curve(
    y_true_test,
    y_probability_test
)

auc_value = roc_auc_score(
    y_true_test,
    y_probability_test
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f'ROC-AUC = {auc_value:.4f}'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Random Classifier'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 18. Prediction Probability Distribution

The model produces a probability between 0 and 1 for each test observation.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=predictions,
    x='probability',
    hue='actual',
    bins=30,
    kde=True,
    stat='density',
    common_norm=False
)

plt.xlabel('Predicted Probability')
plt.ylabel('Density')
plt.title('Prediction Probability Distribution')
plt.tight_layout()
plt.show()

# 19. Threshold Analysis

The model produces probabilities, but a threshold is required to convert probabilities into binary predictions.

Thresholds from **0.20 to 0.80** were evaluated.

## Threshold 0.50

- Accuracy = **70.48%**
- Precision = **31.54%**
- Recall = **79.49%**
- F1 Score = **45.17%**

## Threshold 0.65

- Accuracy = **79.22%**
- Precision = **38.59%**
- Recall = **60.66%**
- F1 Score = **47.17%**

Increasing the threshold reduces the number of positive predictions and changes the precision-recall trade-off.

The threshold sweep is presented as a sensitivity analysis. The 0.50 test evaluation remains the primary reported result.

In [ ]:
threshold_df = pd.read_csv(
    '../models/threshold_analysis.csv'
)

display(threshold_df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_df['Threshold'],
    threshold_df['Precision'],
    marker='o',
    label='Precision'
)

plt.plot(
    threshold_df['Threshold'],
    threshold_df['Recall'],
    marker='o',
    label='Recall'
)

plt.plot(
    threshold_df['Threshold'],
    threshold_df['F1_Score'],
    marker='o',
    label='F1 Score'
)

plt.axvline(
    x=0.50,
    linestyle='--',
    label='Threshold = 0.50'
)

plt.xlabel('Prediction Threshold')
plt.ylabel('Metric Value')
plt.title('Threshold Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 20. Critical Analysis

## Model Strengths

### Class Discrimination

The test ROC-AUC was **0.8167**, indicating that the model has meaningful class-discrimination ability.

### Diabetes Recall

At threshold 0.50, diabetes recall was **79.49%**.

### Class Imbalance Handling

A positive-class weight of approximately **5.5382** was used with BCEWithLogitsLoss.

### Regularization

Dropout, batch normalization, and early stopping were used to improve generalization and training stability.

### Validation and Test Performance

Validation ROC-AUC was **81.95%** and test ROC-AUC was **81.67%**, showing similar ranking performance.

## Limitations

- Diabetes-class precision was relatively low at **31.54%**.
- The model produced **9,082 false positives** at threshold 0.50.
- The model produced **1,080 false negatives**.
- The dataset is survey-based and does not represent a complete clinical assessment.
- External validation was not performed.
- Training was performed on CPU.
- Performance changes with the probability threshold.

## Precision-Recall Trade-off

Lower thresholds identify more positive cases but increase false positives. Higher thresholds make positive predictions more selective but can increase false negatives.

The appropriate threshold depends on the intended application and the relative consequences of different error types.

# 21. Key Insights

## Dataset

- 253,680 original records
- 21 predictive features
- 24,206 duplicate rows
- No missing values
- Imbalanced target distribution

## Preprocessing

- Exact duplicates were removed.
- Stratified sampling was used.
- The split was 70% / 15% / 15%.
- StandardScaler was fitted only on training data.

## Deep Learning

- Architecture: **21 → 64 → 32 → 16 → 1**
- Batch normalization was used.
- Dropout rates were 0.30 and 0.20.
- Class-weighted BCEWithLogitsLoss was used.
- AdamW was used for optimization.
- Early stopping was applied.
- Best checkpoint: epoch 13.

## Final Test Results

- Accuracy: **70.48%**
- Precision: **31.54%**
- Recall: **79.49%**
- F1 Score: **45.17%**
- ROC-AUC: **81.67%**

## Main Learning

For imbalanced classification, accuracy alone does not provide a complete picture. Precision, recall, F1 score, ROC-AUC, confusion matrix, and threshold analysis should be considered together.

# 22. Conclusion

This project presents an end-to-end deep learning workflow for binary diabetes classification using the CDC Diabetes Health Indicators dataset.

A PyTorch-based Multilayer Perceptron was developed using a **21 → 64 → 32 → 16 → 1** architecture.

The model incorporated batch normalization, ReLU activation, dropout regularization, class-weighted binary classification loss, AdamW optimization, learning-rate scheduling, and early stopping.

On the unseen test set, the model achieved:

- **70.48% Accuracy**
- **31.54% Precision**
- **79.49% Recall**
- **45.17% F1 Score**
- **81.67% ROC-AUC**

The project demonstrates the importance of combining deep-learning architecture design, appropriate preprocessing, class imbalance handling, multiple evaluation metrics, and critical analysis.

The model should be considered an academic predictive modeling exercise rather than a clinical diagnostic system.

# 23. Reproducibility

## Environment

- Python 3.14
- PyTorch
- NumPy
- Pandas
- Scikit-learn
- Matplotlib
- Seaborn
- Jupyter

## Random Seed

`42`

## Hardware

Training was performed on CPU because CUDA was unavailable.

## Pipeline

```text
Raw Dataset
    ↓
Data Understanding
    ↓
Exploratory Data Analysis
    ↓
Duplicate Removal
    ↓
Train / Validation / Test Split
    ↓
StandardScaler
    ↓
PyTorch Tensors
    ↓
Class-Weighted MLP
    ↓
Training + Validation
    ↓
Best Model
    ↓
Test Evaluation
    ↓
Threshold Analysis
    ↓
Critical Analysis
    ↓
Conclusion
```

# 24. Project Structure

```text
Week_5_Deep_Learning/
│
├── data/
│   ├── diabetes_binary_health_indicators_BRFSS2015.csv
│   ├── download_dataset.py
│   └── processed/
│       ├── train.csv
│       ├── validation.csv
│       └── test.csv
│
├── models/
│   ├── diabetes_mlp_best.pt
│   ├── training_history.csv
│   ├── training_config.json
│   ├── test_results.csv
│   ├── test_predictions.csv
│   ├── threshold_analysis.csv
│   └── threshold_analysis_summary.json
│
├── src/
│   ├── 01_eda.py
│   ├── 02_preprocess.py
│   ├── 03_train_model.py
│   ├── 04_evaluate_model.py
│   ├── 05_threshold_analysis.py
│   └── 06_create_notebook.py
│
├── visualizations/
│   ├── 01_target_distribution.png
│   ├── 02_feature_distributions.png
│   ├── 03_correlation_heatmap.png
│   ├── 04_top_target_correlations.png
│   ├── 05_class_distribution_pie.png
│   ├── 07_training_validation_loss.png
│   ├── 08_validation_metrics.png
│   ├── 09_confusion_matrix.png
│   ├── 10_roc_curve.png
│   ├── 11_prediction_probability_distribution.png
│   └── 12_threshold_analysis.png
│
└── notebooks/
    └── Diabetes_Deep_Learning_Analysis.ipynb
```